# Week 6: Price Prediction Pipeline – Wakanda Chimobi Ogudu

Generate a product dataset with an LLM, then evaluate price predictions (MAE, RMSE, loss). Uses **OpenRouter**; data is synthetic.

In [ ]:
pip install -q datasets pandas numpy scikit-learn litellm python-dotenv tqdm matplotlib gdown

In [ ]:
import os
import re
import json
import random
from dataclasses import dataclass
from typing import Optional, List, Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from tqdm.auto import tqdm

from dotenv import load_dotenv
load_dotenv(override=True)

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def get_openrouter_api_key():
    if IN_COLAB:
        return userdata.get("OPENROUTER_API_KEY")
    return os.getenv("OPENROUTER_API_KEY")

OPENROUTER_API_KEY = get_openrouter_api_key()
if OPENROUTER_API_KEY:
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
    print(f"OpenRouter API key loaded (starts with {OPENROUTER_API_KEY[:8]}...)")
else:
    print("Warning: OPENROUTER_API_KEY not set. Set it in .env (local) or Colab Secrets.")

try:
    from litellm import completion
except ImportError:
    completion = None

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

In [ ]:
RANDOM_SEED = 42
MIN_PRICE = 0.5
MAX_PRICE = 999.49
MIN_DESCRIPTION_CHARS = 400
MAX_TEXT_EACH = 2000
MAX_TEXT_TOTAL = 3500
TRAIN_FRAC = 0.8
VAL_FRAC = 0.1
TEST_FRAC = 0.1
EVAL_SAMPLE_SIZE = 150
FRONTIER_MODEL = "openrouter/openai/gpt-4.1-nano"

TARGET_DATASET_SIZE = 5000
PRODUCTS_PER_BATCH = 40
GENERATOR_MODEL = "openrouter/openai/gpt-4.1-nano"
PRODUCT_CATEGORIES = [
    "Electronics", "Home & Kitchen", "Appliances", "Sports", "Toys", "Tools", "Beauty", "Books"
]

# Size for one-time export: generate 20k products with the transformer (OpenRouter) and save to CSV / Google Drive.
EXPORT_DATASET_SIZE = 5000

# Static data: set True to use a CSV (or HuggingFace) instead of generating products with the LLM.
USE_STATIC_DATA = True
# Path to CSV, or a Google Drive share link (https://drive.google.com/...). Use repo sample or your Drive file.
STATIC_CSV_PATH = "products_20k.csv"
# If using Colab and saving to Drive, folder name under My Drive (optional).
GOOGLE_DRIVE_EXPORT_FOLDER = "llm_engineering"
# Optional: use HuggingFace dataset instead of CSV (set USE_HF_STATIC = True and leave STATIC_CSV_PATH unused).
USE_HF_STATIC = False
STATIC_HF_DATASET = "ed-donner/pricer-data"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
@dataclass
class ProductItem:
    """A single product with title, description text, and price."""
    title: str
    category: str
    price: float
    full: Optional[str] = None
    weight: float = 0.0
    summary: Optional[str] = None
    item_id: Optional[int] = None

    def __repr__(self):
        short = self.title[:40] + "..." if len(self.title) > 40 else self.title
        return f"<ProductItem {short} = ${self.price:.2f}>"

In [ ]:
REMOVALS = [
    "Part Number", "Best Sellers Rank", "Batteries Included?",
    "Batteries Required?", "Item model number",
]

def simplify_text(text_list) -> str:
    s = str(text_list).replace("\n", " ").replace("\r", "").replace("\t", " ")
    return " ".join(s.split())[:MAX_TEXT_EACH]

def get_weight_from_details(details: dict) -> float:
    weight_str = details.get("Item Weight")
    if not weight_str:
        return 0.0
    parts = weight_str.split()
    if len(parts) < 2:
        return 0.0
    try:
        amount = float(parts[0])
    except ValueError:
        return 0.0
    unit = parts[1].lower()
    if unit == "pounds":
        return amount
    if unit == "ounces":
        return amount / 16
    if unit == "grams":
        return amount / 453.592
    if unit == "kilograms":
        return amount * 2.205
    return 0.0

def scrub_description(title: str, description, features, details: dict) -> str:
    for key in REMOVALS:
        details.pop(key, None)
    result = title + "\n"
    if description:
        result += simplify_text(description) + "\n"
    if features:
        result += simplify_text(features) + "\n"
    if details:
        result += json.dumps(details) + "\n"
    pattern = r"\b(?=[A-Z0-9]{7,}\b)(?=.*[A-Z])(?=.*\d)[A-Z0-9]+\b"
    return re.sub(pattern, "", result).strip()[:MAX_TEXT_TOTAL]

def parse_raw_product(datapoint: dict, category: str) -> Optional[ProductItem]:
    try:
        price = float(datapoint["price"])
    except (ValueError, TypeError, KeyError):
        return None
    if not (MIN_PRICE <= price <= MAX_PRICE):
        return None
    title = datapoint.get("title", "")
    if not title:
        return None
    description = datapoint.get("description") or []
    features = datapoint.get("features") or []
    try:
        details = json.loads(datapoint["details"]) if isinstance(datapoint.get("details"), str) else {}
    except (json.JSONDecodeError, TypeError):
        details = {}
    weight = get_weight_from_details(details)
    full = scrub_description(title, description, features, dict(details))
    if len(full) < MIN_DESCRIPTION_CHARS:
        return None
    return ProductItem(
        title=title,
        category=category,
        price=price,
        full=full,
        weight=weight,
        summary=full,
    )

In [ ]:
GENERATION_SYSTEM_PROMPT = """You are a data generator. Output only valid JSON, no markdown or explanation.
Generate a JSON array of product objects. Each object must have exactly: title (string), category (string), description (string, at least 100 chars), price (number, USD between 0.5 and 999.49).
Use realistic product names and descriptions. Vary categories and prices. Output only the JSON array."""

def generate_one_batch(
    batch_size: int,
    categories: List[str],
    model: str = GENERATOR_MODEL,
) -> List[dict]:
    """Ask the frontier model to generate one batch of product records as JSON."""
    if completion is None:
        return []
    cats_str = ", ".join(categories)
    user_msg = f"Generate exactly {batch_size} different products. Categories to use: {cats_str}. Output a JSON array of objects with keys: title, category, description, price."
    messages = [
        {"role": "system", "content": GENERATION_SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    try:
        response = completion(model=model, messages=messages)
        text = (response.choices[0].message.content or "").strip()
        text = re.sub(r"^```\w*\n?", "", text).strip()
        text = re.sub(r"\n?```\s*$", "", text).strip()
        data = json.loads(text)
        if not isinstance(data, list):
            data = [data]
        return data
    except (json.JSONDecodeError, Exception) as e:
        print(f"Batch parse error: {e}")
        return []

def generated_record_to_item(record: dict, item_id: int) -> Optional[ProductItem]:
    """Convert a generated record (title, category, description, price) to ProductItem."""
    try:
        price = float(record.get("price", 0))
    except (ValueError, TypeError):
        return None
    if not (MIN_PRICE <= price <= MAX_PRICE):
        return None
    title = (record.get("title") or "").strip()
    if not title:
        return None
    category = (record.get("category") or "Other").strip()
    description = (record.get("description") or title).strip()
    if len(description) < 50:
        description = description + " " + (title * 3)[:200]
    full = (title + "\n" + description)[:MAX_TEXT_TOTAL]
    return ProductItem(
        title=title,
        category=category,
        price=price,
        full=full,
        weight=0.0,
        summary=full,
        item_id=item_id,
    )

def generate_product_dataset(
    target_size: int = TARGET_DATASET_SIZE,
    batch_size: int = PRODUCTS_PER_BATCH,
    categories: Optional[List[str]] = None,
    model: str = GENERATOR_MODEL,
) -> List[ProductItem]:
    """Generate a dataset of the same size using the frontier model."""
    categories = categories or PRODUCT_CATEGORIES
    items = []
    num_batches = (target_size + batch_size - 1) // batch_size
    for batch_i in tqdm(range(num_batches), desc="Generating products"):
        batch = generate_one_batch(batch_size, categories, model=model)
        for rec in batch:
            item = generated_record_to_item(rec, len(items))
            if item is not None:
                items.append(item)
        if len(items) >= target_size:
            break
    for idx, item in enumerate(items):
        item.item_id = idx
    return items[:target_size]

In [ ]:
import csv
# --- One-time: generate 20k products with the transformer (OpenRouter) and save to CSV + Google Drive ---
def product_item_to_csv_row(item: ProductItem) -> tuple:
    """Format ProductItem as (text_block, price) for human_out-style CSV."""
    desc = (item.full or item.summary or item.title or "").strip()
    text = f"Title: {item.title}\nCategory: {item.category}\nDescription: {desc}\nDetails: \n"
    return (text, item.price)

def save_products_to_csv(items: List[ProductItem], filepath: str) -> None:
    """Write products to CSV (human_out format)."""
    with open(filepath, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        for item in items:
            writer.writerow(product_item_to_csv_row(item))
    print(f"Saved {len(items):,} products to {filepath}")

# Generate 20k products using the frontier model (transformer via OpenRouter).
items_20k = generate_product_dataset(
    target_size=EXPORT_DATASET_SIZE,
    batch_size=PRODUCTS_PER_BATCH,
    categories=PRODUCT_CATEGORIES,
    model=GENERATOR_MODEL,
)
print(f"Generated {len(items_20k):,} products")

# Save to local CSV.
EXPORT_CSV_NAME = "products_20k.csv"
save_products_to_csv(items_20k, EXPORT_CSV_NAME)

# Optionally save to Google Drive (Colab: mount Drive first in the next cell).
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        import os
        dest_dir = f"/content/drive/MyDrive/{GOOGLE_DRIVE_EXPORT_FOLDER}"
        os.makedirs(dest_dir, exist_ok=True)
        dest_path = f"{dest_dir}/{EXPORT_CSV_NAME}"
        import shutil
        shutil.copy(EXPORT_CSV_NAME, dest_path)
        print(f"Copied to Google Drive: {dest_path}")
        print("Share the file (Right-click > Share) and use the link as STATIC_CSV_PATH to load later.")
    except Exception as e:
        print(f"Drive save skipped: {e}. Run 'from google.colab import drive; drive.mount(\"/content/drive\")' first.")
else:
    print("Local only. Upload products_20k.csv to Google Drive, then set STATIC_CSV_PATH to the file's share link.")

In [ ]:
from pathlib import Path

def _parse_static_row_text(text: str) -> tuple:
    """Parse 'Title: ... Category: ... Description: ... Details: ...' block; return (title, category, full)."""
    title = category = None
    for line in text.strip().split("\n"):
        line = line.strip()
        if line.startswith("Title:"):
            title = line[6:].strip()
        elif line.startswith("Category:"):
            category = line[9:].strip()
    full = (text or "").strip()[:MAX_TEXT_TOTAL]
    return (title or "Product", category or "Other", full)

def load_static_products_from_csv(csv_path: str) -> List[ProductItem]:
    """Load products from a CSV (local path or Google Drive share URL). Column 0 = product text, column 1 = price."""
    path = Path(csv_path)
    is_drive_url = "drive.google.com" in str(csv_path)
    if is_drive_url:
        try:
            import gdown
            import tempfile
            with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
                tmp_path = tmp.name
            gdown.download(csv_path, tmp_path, quiet=True, fuzzy=True)
            path = Path(tmp_path)
        except Exception as e:
            raise FileNotFoundError(f"Could not download from Google Drive: {e}") from e
    elif not path.exists() and not path.is_absolute():
        path = Path.cwd() / csv_path
    if not path.is_absolute():
        path = path.resolve()
    if not path.exists():
        path = Path(csv_path).resolve()
    print(f"Loading static CSV from: {path} (exists: {path.exists()})")
    items = []
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for i, row in enumerate(reader):
            if len(row) < 2:
                continue
            text, price_str = row[0], row[1]
            try:
                price = float(price_str)
            except (ValueError, TypeError):
                continue
            if not (MIN_PRICE <= price <= MAX_PRICE):
                continue
            title, category, full = _parse_static_row_text(text)
            if len(full) < 50:
                full = f"{title}\n{full}"
            items.append(ProductItem(title=title, category=category, price=price, full=full, weight=0.0, summary=full, item_id=i))
    if is_drive_url and path.exists():
        path.unlink()
    return items

def load_static_products_from_hf(dataset_name: str) -> List[ProductItem]:
    """Load products from HuggingFace (e.g. ed-donner/pricer-data). Concatenates train + test."""
    from datasets import load_dataset as hf_load
    ds = hf_load(dataset_name)
    items = []
    for split in ("train", "test", "validation"):
        if split not in ds:
            continue
        for i, row in enumerate(ds[split]):
            try:
                price = float(row.get("price", 0))
            except (ValueError, TypeError):
                continue
            if not (MIN_PRICE <= price <= MAX_PRICE):
                continue
            title = (row.get("title") or "Product").strip()
            category = (row.get("category") or "Other").strip()
            full = (row.get("full") or row.get("summary") or row.get("text") or title or "")
            if isinstance(full, list):
                full = " ".join(str(x) for x in full)
            full = str(full).strip()[:MAX_TEXT_TOTAL] or title
            items.append(ProductItem(title=title, category=category, price=price, full=full, weight=0.0, summary=full, item_id=len(items)))
    return items

def load_static_products() -> List[ProductItem]:
    """Load static product list from CSV or HuggingFace according to notebook constants."""
    if USE_HF_STATIC:
        return load_static_products_from_hf(STATIC_HF_DATASET)
    return load_static_products_from_csv(STATIC_CSV_PATH)

In [ ]:
# Use static data (CSV or HuggingFace) or generate products with the LLM.
if USE_STATIC_DATA:
    all_items = load_static_products()
    print(f"Loaded {len(all_items):,} products from static data ({STATIC_CSV_PATH if not USE_HF_STATIC else STATIC_HF_DATASET})")
else:
    all_items = generate_product_dataset(
        target_size=TARGET_DATASET_SIZE,
        batch_size=PRODUCTS_PER_BATCH,
        categories=PRODUCT_CATEGORIES,
        model=GENERATOR_MODEL,
    )
    print(f"Generated {len(all_items):,} products (target {TARGET_DATASET_SIZE:,})")
if all_items:
    print(f"Example: {all_items[0]}")

In [ ]:
def train_val_test_split(
    items: List[ProductItem],
    train_frac: float = TRAIN_FRAC,
    val_frac: float = VAL_FRAC,
    test_frac: float = TEST_FRAC,
    seed: int = RANDOM_SEED,
) -> tuple:
    n = len(items)
    indices = list(range(n))
    random.Random(seed).shuffle(indices)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    n_test = n - n_train - n_val  # remainder so no item is dropped
    train_idx = indices[:n_train]
    val_idx = indices[n_train : n_train + n_val]
    test_idx = indices[n_train + n_val :]
    train_items = [items[i] for i in train_idx]
    val_items = [items[i] for i in val_idx]
    test_items = [items[i] for i in test_idx]
    return train_items, val_items, test_items

train_items, val_items, test_items = train_val_test_split(all_items)
print(f"Train: {len(train_items):,}  Val: {len(val_items):,}  Test: {len(test_items):,}")

In [ ]:
def parse_price_from_model_response(response: str) -> float:
    """Extract a single numeric price from model output (e.g. $12.99 or 12.99)."""
    if response is None:
        return 0.0
    s = str(response).replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.?\d+", s)
    if match:
        return max(0.0, float(match.group()))
    return 0.0

def compute_metrics(actual_prices: List[float], predicted_prices: List[float]) -> dict:
    """Compute MAE, RMSE, MSE (loss), and R²."""
    actual = np.asarray(actual_prices, dtype=float)
    pred = np.asarray(predicted_prices, dtype=float)
    mae = np.mean(np.abs(actual - pred))
    mse = mean_squared_error(actual, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(actual, pred)
    return {"mae": mae, "rmse": rmse, "mse": mse, "r2": r2}

def run_benchmark(
    predictor: Callable,
    test_data: List[ProductItem],
    sample_size: int = EVAL_SAMPLE_SIZE,
    parse_response: bool = True,
    return_arrays: bool = False,
):
    """Run predictor on a sample of test items and return MAE/RMSE metrics. If return_arrays=True, returns (metrics, actuals, preds)."""
    size = min(sample_size, len(test_data))
    subset = random.Random(RANDOM_SEED).sample(test_data, size)
    actuals = []
    preds = []
    for item in tqdm(subset, desc="Evaluating"):
        out = predictor(item)
        if parse_response:
            out = parse_price_from_model_response(out)
        preds.append(float(out))
        actuals.append(item.price)
    metrics = compute_metrics(actuals, preds)
    if return_arrays:
        return metrics, actuals, preds
    return metrics

In [ ]:
def print_metrics(metrics: dict, label: str = "Model"):
    print(f"{label}:")
    print(f"  MAE  = ${metrics['mae']:.2f}")
    print(f"  RMSE = ${metrics['rmse']:.2f}")
    print(f"  Loss (MSE) = {metrics['mse']:.2f}")
    print(f"  R²   = {metrics['r2']:.4f}")

In [ ]:
mean_train_price = np.mean([p.price for p in train_items])

def baseline_constant_pricer(item: ProductItem) -> float:
    return mean_train_price

baseline_metrics, eval_actuals, eval_baseline_preds = run_benchmark(
    baseline_constant_pricer, test_items, parse_response=False, return_arrays=True
)
print_metrics(baseline_metrics, "Baseline (constant = mean price)")

In [ ]:
def build_price_prompt(item: ProductItem) -> list:
    """Build messages for the LLM: estimate price from product description."""
    text = item.summary or item.full or item.title
    return [
        {"role": "user", "content": f"Estimate the price of this product in USD. Respond with only the number (e.g. 29.99).\n\n{text}"}
    ]

def frontier_model_pricer(item: ProductItem, model: str = FRONTIER_MODEL) -> str:
    """Call frontier model to predict price."""
    if completion is None:
        return "0"
    messages = build_price_prompt(item)
    try:
        response = completion(model=model, messages=messages)
        return response.choices[0].message.content or "0"
    except Exception as e:
        print(f"API error: {e}")
        return "0"

In [ ]:
print(f"Benchmarking frontier model via OpenRouter: {FRONTIER_MODEL}")
print("Ensure OPENROUTER_API_KEY is set in .env (or Colab Secrets).")
frontier_metrics, _, eval_frontier_preds = run_benchmark(
    lambda item: frontier_model_pricer(item, model=FRONTIER_MODEL),
    test_items,
    sample_size=EVAL_SAMPLE_SIZE,
    parse_response=True,
    return_arrays=True,
)
print_metrics(frontier_metrics, f"Frontier ({FRONTIER_MODEL})")

In [ ]:
comparison = pd.DataFrame(
    [
        {"Model": "Baseline (mean)", "MAE": baseline_metrics["mae"], "RMSE": baseline_metrics["rmse"], "Loss (MSE)": baseline_metrics["mse"], "R²": baseline_metrics["r2"]},
        {"Model": FRONTIER_MODEL, "MAE": frontier_metrics["mae"], "RMSE": frontier_metrics["rmse"], "Loss (MSE)": frontier_metrics["mse"], "R²": frontier_metrics["r2"]},
    ]
)
print(comparison.to_string(index=False))

In [ ]:
# Plot results and model comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1. Actual vs Predicted (scatter)
ax1 = axes[0]
max_price = max(max(eval_actuals), max(eval_baseline_preds), max(eval_frontier_preds))
ax1.scatter(eval_actuals, eval_baseline_preds, alpha=0.5, s=20, label="Baseline (mean)", color="C1")
ax1.scatter(eval_actuals, eval_frontier_preds, alpha=0.5, s=20, label=f"Frontier ({FRONTIER_MODEL.split('/')[-1]})", color="C0")
ax1.plot([0, max_price], [0, max_price], "k--", alpha=0.7, label="Perfect prediction")
ax1.set_xlabel("Actual price ($)")
ax1.set_ylabel("Predicted price ($)")
ax1.set_title("Actual vs Predicted Price")
ax1.legend()
ax1.set_xlim(0, max_price * 1.02)
ax1.set_ylim(0, max_price * 1.02)
ax1.set_aspect("equal")

# 2. Metrics comparison (bar charts)
ax2 = axes[1]
models = ["Baseline (mean)", FRONTIER_MODEL.split("/")[-1]]
x = np.arange(len(models))
width = 0.25
mae_vals = [baseline_metrics["mae"], frontier_metrics["mae"]]
rmse_vals = [baseline_metrics["rmse"], frontier_metrics["rmse"]]
loss_vals = [baseline_metrics["mse"], frontier_metrics["mse"]]
bars1 = ax2.bar(x - width, mae_vals, width, label="MAE ($)", color="C2")
bars2 = ax2.bar(x, rmse_vals, width, label="RMSE ($)", color="C3")
ax2_twin = ax2.twinx()
bars3 = ax2_twin.bar(x + width, loss_vals, width, label="Loss (MSE, $²)", color="C4", alpha=0.8)
ax2.set_ylabel("Error ($)")
ax2_twin.set_ylabel("Loss (MSE, $²)")
ax2.set_title("MAE, RMSE & Loss by model")
ax2.set_xticks(x)
ax2.set_xticklabels(models)
ax2.legend(loc="upper left")
ax2_twin.legend(loc="upper right")
ax2.bar_label(bars1, fmt="$%.0f")
ax2.bar_label(bars2, fmt="$%.0f")
ax2_twin.bar_label(bars3, fmt="%.0f")
# R² in title or as text (scale differs from MAE/RMSE)
r2_text = f"R²: Baseline={baseline_metrics['r2']:.3f}, Frontier={frontier_metrics['r2']:.3f}"
ax2.set_xlabel(r2_text, fontsize=9)

plt.tight_layout()
plt.show()